In [ ]:
import xgboost as xgb

xgb_train = xgb.DMatrix(xtrain_w2v,label=ytrain)
xgb_val = xgb.DMatrix(xtest_w2v, label=ytest)
xgb_test = xgb.DMatrix(test_w2v)



In [ ]:
params = {

      'objective' : 'binary:logistic',
      'max_depth' : 6,
      'min_child_weight' : 1,
      'eta' : 0.3,
      'subsample' : 1,
      'colsample_bytree' : 1
}

def f1_eval(preds,xgb_train):
  y_true = xgb_train.get_label().astype(int)
  preds_binary = (preds >= 0.3).astype(int)
  return 'f1_score',f1_score(y_true,preds_binary)


In [ ]:

GridSearch = [
    (max_depth, min_child_weight)
for max_depth in range(6,10)
  for min_child_weight in range(5,8)
]



max_f1 = 0
best_parms = None

for max_depth,min_child_weight in GridSearch:
  print(" CV with max_depht={} and min_child_weight={} ".format(max_depth,min_child_weight))

  # update params
  params['max_depth'] = max_depth
  params['min_child_weight'] = min_child_weight

  #cross validation K-Fold
  cv_results = xgb.cv(params, xgb_train, custom_metric=f1_eval, num_boost_round = 200, seed = 16, nfold = 5, early_stopping_rounds=10, maximize=True)


  mean_f1 = cv_results['test-f1_score-mean'].max()
  boost_round = cv_results['test-f1_score-mean'].argmax()

  print("\t F1 score {} for {} round ".format(mean_f1,boost_round))

  if mean_f1 > max_f1:
    max_f1 = mean_f1
    best_parms = (max_depth, min_child_weight)

  print("\n Best params are {} {} and F1 score is {}".format(best_parms[0],best_parms[1],max_f1))


print("Best params are {} {} and F1 score is {}".format(best_parms[0],best_parms[1],max_f1))

params['max_depth'] = best_parms[0]
params['min_child_weight'] = best_parms[1]


 CV with max_depht=6 and min_child_weight=5 
	 F1 score 0.6854406000000001 for 81 round 

 Best params are 6 5 and F1 score is 0.6854406000000001
 CV with max_depht=6 and min_child_weight=6 
	 F1 score 0.6841482000000001 for 105 round 

 Best params are 6 5 and F1 score is 0.6854406000000001
 CV with max_depht=6 and min_child_weight=7 
	 F1 score 0.669352 for 47 round 

 Best params are 6 5 and F1 score is 0.6854406000000001
 CV with max_depht=7 and min_child_weight=5 
	 F1 score 0.6826396000000001 for 71 round 

 Best params are 6 5 and F1 score is 0.6854406000000001
 CV with max_depht=7 and min_child_weight=6 
	 F1 score 0.662411 for 43 round 

 Best params are 6 5 and F1 score is 0.6854406000000001
 CV with max_depht=7 and min_child_weight=7 
	 F1 score 0.6849266 for 85 round 

 Best params are 6 5 and F1 score is 0.6854406000000001
 CV with max_depht=8 and min_child_weight=5 
	 F1 score 0.6770854 for 54 round 

 Best params are 6 5 and F1 score is 0.6854406000000001
 CV with max_de

In [ ]:

# Data & features

GridSearch = [
    (subsample, colsample_bytree)
for subsample in [i/10. for i in range(5,10)]
  for colsample_bytree in [j/10. for j in range(5,10)]
]



max_f1 = 0
best_parms = None

for subsample,colsample_bytree in GridSearch:
  print(" CV with subsample={} and colsample_bytree={} ".format(subsample,colsample_bytree))

  # update params
  params['subsample'] = subsample
  params['colsample_bytree'] = colsample_bytree

  #cross validation K-Fold
  cv_results = xgb.cv(params, xgb_train, custom_metric=f1_eval, num_boost_round = 200, seed = 16, nfold = 5, early_stopping_rounds=10, maximize=True)

  print(cv_results)
  # mean_f1 = cv_results['train-f1_score-mean'].max()
  # boost_round = cv_results['train-f1_score-mean'].argmax()

  # print(" \t F1 score {} for {} round ".format(mean_f1,boost_round))

  # if mean_f1 > max_f1:
  #   max_f1 = mean_f1
  #   best_parms = (subsample, colsample_bytree)


print("Best params are {} {} and F1 score is {}".format(best_parms[0],best_parms[1],max_f1))

params['subsample'] = best_parms[0]
params['colsample_bytree'] = best_parms[1]



In [ ]:
# learning Rate


learning_rate = [.3, .2, .1, .05, .01, .005]

max_f1 = 0
best_parms = None

for rate in learning_rate:
  print(" CV with learning rate ={} ".format(rate))

  # update params
  params['eta'] = rate


  #cross validation K-Fold
  cv_results = xgb.cv(params, xgb_train, custom_metric=f1_eval, num_boost_round = 200, seed = 16, nfold = 5, early_stopping_rounds=20, maximize=True)

  mean_f1 = cv_results['test-f1_score-mean'].max()
  boost_round = cv_results['test-f1_score-mean'].argmax()

  print(" \t F1 score {} for {} round ".format(mean_f1,boost_round))

  if mean_f1 > max_f1:
    max_f1 = mean_f1
    best_parms = rate


print("Best params are {} and F1 score is {}".format(best_parms,max_f1))

params['eta'] = best_parms


In [ ]:

params


In [ ]:
xgb_model = xgb.train(params, xgb_train, custom_metric=f1_eval, num_boost_round = 1000, early_stopping_rounds=20, evals=[(xgb_val,"Validation")],maximize=True)

ypred = xgb_model.predict(xtest)
